# OnomaCap J-text — Seoul pronounced-form transfer

Seoul Corpus의 **한글 `pWord.prono.`**만 canonical Jamo로 분해해 BART를 denoising 사전학습한 뒤 OnomaCap에 전이합니다. 이 단계에서는 Seoul FLAC와 `phoneme` tier를 사용하지 않습니다. 예를 들어 phoneme tier의 `k0 a k7`을 토큰으로 매핑하지 않고, 발음형 `각`을 Unicode NFD로 분해한 `ᄀ ᅡ ᆨ`만 현재 OnomaCap Jamo 토큰으로 사용합니다. OnomaCap 전이 중 HTSAT는 완전히 동결합니다.


In [ ]:
# 의존성 설치
%pip install -q transformers==4.36.2 tokenizers==0.15.2 "huggingface-hub<1.0" torchlibrosa==0.1.0 librosa==0.10.2.post1 ruamel.yaml==0.17.40 gdown==5.2.0 "kagglehub>=0.3.12" pycocoevalcap==1.2 "scikit-learn>=1.4,<1.7" "pandas>=2.1,<2.4" matplotlib seaborn tqdm loguru warmup-scheduler gensim

# 저장소 clone 및 WavCaps overlay 적용
import shutil
from pathlib import Path

ONOMAHOW_REPO = 'https://github.com/youhan200203/OnomaHoW.git'
ONOMAHOW_REF = 'working'
WAVCAPS_REPO = 'https://github.com/XinhaoMei/WavCaps.git'
WAVCAPS_COMMIT = 'a5a9649ce305d7fe82cfcf5d6a4a12f03df9ef1e'
ANNOTATION_REPO = 'https://github.com/jspirit01/sound-to-onomatopoeia.git'

ONOMAHOW_DIR = Path('/content/OnomaHoW')
WAVCAPS_DIR = Path('/content/WavCaps')
ANNOTATION_DIR = Path('/content/sound-to-onomatopoeia')
for transient_dir in (ONOMAHOW_DIR, WAVCAPS_DIR, ANNOTATION_DIR):
    if transient_dir.exists():
        shutil.rmtree(transient_dir)

!git clone -q --depth 1 --branch "{ONOMAHOW_REF}" "{ONOMAHOW_REPO}" "{ONOMAHOW_DIR}"
!git clone -q "{WAVCAPS_REPO}" "{WAVCAPS_DIR}"
!git -C "{WAVCAPS_DIR}" checkout --detach -q "{WAVCAPS_COMMIT}"
!git clone -q --depth 1 "{ANNOTATION_REPO}" "{ANNOTATION_DIR}"
assert (ONOMAHOW_DIR / '.git').is_dir()
assert (WAVCAPS_DIR / '.git').is_dir()
assert (ANNOTATION_DIR / '.git').is_dir()

overlay_root = ONOMAHOW_DIR / 'wavcaps_patch'
overlay_files = sorted(overlay_root.rglob('*.py'))
assert overlay_files, 'wavcaps_patch overlay가 비어 있습니다.'
for source in overlay_files:
    destination = WAVCAPS_DIR / source.relative_to(overlay_root)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    assert source.read_bytes() == destination.read_bytes()

helper_destination = WAVCAPS_DIR / 'captioning/tools/jamo_preprocessing.py'
config_destination = WAVCAPS_DIR / 'captioning/settings/onomacap_jamo.yaml'
shutil.copy2(ONOMAHOW_DIR / 'jamo_preprocessing.py', helper_destination)
shutil.copy2(ONOMAHOW_DIR / 'configs/onomacap_jamo.yaml', config_destination)
checked_out_commit_output = !git -C "{WAVCAPS_DIR}" rev-parse HEAD
assert len(checked_out_commit_output) == 1, checked_out_commit_output
checked_out_commit = checked_out_commit_output[0].strip()
assert checked_out_commit == WAVCAPS_COMMIT
print(f'WavCaps {checked_out_commit} + {len(overlay_files)} overlay files')


In [ ]:
# 공통 설정과 canonical Jamo 정의
import csv
import gc
import hashlib
import json
import math
import os
import random
import re
import sys
import tarfile
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from google.colab import drive

MODEL_SEED = 20
SPLIT_SEED = 20
EVAL_BEAM_SIZE = 3
MAX_LENGTH = 80
EXPERIMENT_NAME = 'onomacap_jamo_seoul_jtext_v1'
CHOSEONG = tuple(chr(codepoint) for codepoint in range(0x1100, 0x1113))
JUNGSEONG = tuple(chr(codepoint) for codepoint in range(0x1161, 0x1176))
JONGSEONG = tuple(chr(codepoint) for codepoint in range(0x11A8, 0x11C3))
JAMO_VOCAB = CHOSEONG + JUNGSEONG + JONGSEONG
JAMO_SET = frozenset(JAMO_VOCAB)
assert len(JAMO_VOCAB) == 67

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(MODEL_SEED)
drive.mount('/content/drive')
SEOUL_DRIVE_ROOT = Path('/content/drive/MyDrive/OnomaCap/Seoul-Corpus')
JTEXT_DRIVE_ROOT = Path('/content/drive/MyDrive/OnomaCap/seoul_jtext')
JTEXT_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
assert (SEOUL_DRIVE_ROOT / 'label.tgz').is_file()


In [ ]:
# label.tgz -> label-TextGrid.zip -> TextGrid (sound.tgz는 J-text에서 사용하지 않음)
SEOUL_WORK_ROOT = Path('/content/seoul_jtext_data')
if SEOUL_WORK_ROOT.exists():
    shutil.rmtree(SEOUL_WORK_ROOT)
OUTER_DIR = SEOUL_WORK_ROOT / 'outer'
TEXTGRID_DIR = SEOUL_WORK_ROOT / 'textgrids'
OUTER_DIR.mkdir(parents=True)
TEXTGRID_DIR.mkdir(parents=True)

def safe_destination(root, member_name):
    root = root.resolve()
    destination = (root / member_name).resolve()
    if destination != root and root not in destination.parents:
        raise ValueError(f'Unsafe archive member: {member_name!r}')
    return destination

with tarfile.open(SEOUL_DRIVE_ROOT / 'label.tgz', 'r:*') as archive:
    for member in archive.getmembers():
        safe_destination(OUTER_DIR, member.name)
    archive.extractall(OUTER_DIR)

nested_zips = sorted(OUTER_DIR.rglob('label-TextGrid.zip'))
if len(nested_zips) != 1:
    nested_zips = sorted(OUTER_DIR.rglob('*TextGrid*.zip'))
assert len(nested_zips) == 1, nested_zips
with zipfile.ZipFile(nested_zips[0]) as archive:
    for member in archive.infolist():
        safe_destination(TEXTGRID_DIR, member.filename)
    archive.extractall(TEXTGRID_DIR)

textgrid_paths = sorted(TEXTGRID_DIR.rglob('*.TextGrid'))
assert textgrid_paths, 'TextGrid가 없습니다.'
assert not list(SEOUL_WORK_ROOT.rglob('*.flac')), 'J-text 경로에 FLAC가 섞였습니다.'
print('nested zip:', nested_zips[0])
print('TextGrid files:', len(textgrid_paths))


In [ ]:
# UTF-16 TextGrid ordered-tier parser: 중복 tier 이름을 보존한다.
EXPECTED_TIER_NAMES = (
    'phoneme',
    'pWord.prono.',
    'pWord.prono.',
    'utt.prono.',
    'pWord.ortho.',
    'pWord.ortho.',
    'utt.ortho.',
)
ITEM_RE = re.compile(r'(?m)^\s*item \[(\d+)\]:\s*$')
NAME_RE = re.compile(r'(?m)^\s*name = "([^"]*)"\s*$')
INTERVAL_RE = re.compile(
    r'intervals \[(\d+)\]:\s*'
    r'xmin = ([0-9.eE+-]+)\s*'
    r'xmax = ([0-9.eE+-]+)\s*'
    r'text = "([^"]*)"',
    re.S,
)
HANGUL_WORD_RE = re.compile(r'^[가-힣 ]+$')
EVENT_RE = re.compile(r'^<[^>]+>$')
SPEAKER_FILE_RE = re.compile(r'^(?P<speaker>.+)f\d+$')

@dataclass(frozen=True)
class TextGridInterval:
    index: int
    xmin: float
    xmax: float
    text: str

@dataclass(frozen=True)
class TextGridTier:
    name: str
    intervals: tuple

def read_textgrid_ordered(path):
    raw_bytes = Path(path).read_bytes()
    if raw_bytes.startswith((b'\xfe\xff', b'\xff\xfe')):
        text = raw_bytes.decode('utf-16')
    else:
        text = raw_bytes.decode('utf-8-sig')
    matches = list(ITEM_RE.finditer(text))
    tiers = []
    for offset, match in enumerate(matches):
        end = matches[offset + 1].start() if offset + 1 < len(matches) else len(text)
        block = text[match.end():end]
        name_match = NAME_RE.search(block)
        if name_match is None:
            raise ValueError(f'Missing tier name: {path}, item={match.group(1)}')
        intervals = tuple(
            TextGridInterval(int(index), float(xmin), float(xmax), value)
            for index, xmin, xmax, value in INTERVAL_RE.findall(block)
        )
        tiers.append(TextGridTier(name_match.group(1), intervals))
    names = tuple(tier.name for tier in tiers)
    if names != EXPECTED_TIER_NAMES:
        raise ValueError(f'Unexpected tier order in {path}: {names!r}')
    hangul_values = [item.text for item in tiers[1].intervals if HANGUL_WORD_RE.fullmatch(item.text)]
    roman_values = [item.text for item in tiers[2].intervals if not EVENT_RE.fullmatch(item.text)]
    if not hangul_values or any(re.search(r'[가-힣]', value) for value in roman_values):
        raise ValueError(f'Hangul/romanized pWord tiers are reversed or malformed: {path}')
    if len(tiers[1].intervals) != len(tiers[2].intervals):
        raise ValueError(f'Pronounced pWord tier length mismatch: {path}')
    return tuple(tiers)

def speaker_id_from_path(path):
    match = SPEAKER_FILE_RE.fullmatch(Path(path).stem)
    if match is None:
        raise ValueError(f'Unknown Seoul filename: {Path(path).name}')
    return match.group('speaker')

def pronounced_hangul_to_jamo(text):
    normalized = unicodedata.normalize('NFC', str(text))
    compact = ''.join(normalized.split())
    if not compact or not all(0xAC00 <= ord(character) <= 0xD7A3 for character in compact):
        raise ValueError(f'Not a precomposed-Hangul pronounced form: {text!r}')
    decomposed = unicodedata.normalize('NFD', compact)
    if not decomposed or set(decomposed) - JAMO_SET:
        raise ValueError(f'Unsupported canonical Jamo in {text!r}: {decomposed!r}')
    reconstructed = unicodedata.normalize('NFC', decomposed)
    if reconstructed != compact:
        raise ValueError(f'Jamo round-trip failed: {text!r} -> {reconstructed!r}')
    return ' '.join(decomposed)

# phoneme 표기는 현재 자모 tokenizer와 다른 체계이며 J-text 입력으로 사용하지 않는다.
assert pronounced_hangul_to_jamo('각') == 'ᄀ ᅡ ᆨ'
assert set('k0 a k7'.split()).isdisjoint(JAMO_SET)
sample_tiers = read_textgrid_ordered(textgrid_paths[0])
print(tuple(tier.name for tier in sample_tiers))
print('phoneme example (excluded):', [item.text for item in sample_tiers[0].intervals[:8]])
print('Hangul pWord.prono. examples:', [item.text for item in sample_tiers[1].intervals[:12]])


In [ ]:
# 전체 한글 pWord.prono. 추출과 품질 검증
records = []
event_counts = Counter()
rejected_counts = Counter()
too_long = []
for path in textgrid_paths:
    tiers = read_textgrid_ordered(path)
    speaker_id = speaker_id_from_path(path)
    # tiers[0] phoneme 및 tiers[2] romanized pWord는 의도적으로 사용하지 않는다.
    for interval in tiers[1].intervals:
        value = interval.text.strip()
        if EVENT_RE.fullmatch(value):
            event_counts[value] += 1
            continue
        if not HANGUL_WORD_RE.fullmatch(value):
            rejected_counts['non_hangul'] += 1
            continue
        jamo = pronounced_hangul_to_jamo(value)
        jamo_length = len(jamo.split())
        if jamo_length + 2 > MAX_LENGTH:
            too_long.append((path.name, interval.index, value, jamo_length))
            continue
        records.append({
            'source_textgrid': path.name,
            'speaker_id': speaker_id,
            'interval_index': interval.index,
            'xmin': interval.xmin,
            'xmax': interval.xmax,
            'pronounced_hangul': ''.join(value.split()),
            'canonical_jamo': jamo,
            'jamo_length': jamo_length,
        })

assert records
assert not rejected_counts, rejected_counts
assert not too_long, too_long[:10]
assert all(set(record['canonical_jamo'].split()) <= JAMO_SET for record in records)
assert all(
    unicodedata.normalize('NFC', ''.join(record['canonical_jamo'].split())) == record['pronounced_hangul']
    for record in records
)
speaker_ids = sorted({record['speaker_id'] for record in records})
lengths = np.asarray([record['jamo_length'] for record in records])
processing_report = {
    'textgrid_files': len(textgrid_paths),
    'speakers': len(speaker_ids),
    'accepted_pwords': len(records),
    'unique_pwords': len({record['pronounced_hangul'] for record in records}),
    'jamo_length_median': float(np.median(lengths)),
    'jamo_length_p95': float(np.percentile(lengths, 95)),
    'jamo_length_max': int(lengths.max()),
    'events_excluded': dict(event_counts.most_common()),
    'phoneme_tier_used_for_training': False,
    'romanized_pword_tier_used_for_training': False,
    'flac_used_for_training': False,
}
display(processing_report)
display(pd.DataFrame(records).head())


In [ ]:
# 화자 단위 80/20 train/validation split (Seoul test는 만들지 않는다.)
SPEAKER_META_RE = re.compile(r'^s\d+(?P<gender>[mf])(?P<age>\d+)$')
strata = defaultdict(list)
for speaker_id in speaker_ids:
    match = SPEAKER_META_RE.fullmatch(speaker_id)
    if match is None:
        raise ValueError(f'Cannot parse Seoul speaker metadata: {speaker_id!r}')
    decade = int(match.group('age')) // 10
    strata[(match.group('gender'), decade)].append(speaker_id)

split_rng = random.Random(SPLIT_SEED)
train_speakers, val_speakers = set(), set()
for stratum, members in sorted(strata.items()):
    members = sorted(members)
    split_rng.shuffle(members)
    val_count = max(1, round(len(members) * 0.2))
    val_speakers.update(members[:val_count])
    train_speakers.update(members[val_count:])
assert train_speakers and val_speakers and train_speakers.isdisjoint(val_speakers)
train_records = [record for record in records if record['speaker_id'] in train_speakers]
val_records = [record for record in records if record['speaker_id'] in val_speakers]
assert len(train_records) + len(val_records) == len(records)

def write_jsonl(path, rows):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + '\n')
    os.replace(temporary, path)

TRAIN_JSONL = JTEXT_DRIVE_ROOT / 'seoul_jtext_train.jsonl'
VAL_JSONL = JTEXT_DRIVE_ROOT / 'seoul_jtext_val.jsonl'
SPLIT_MANIFEST = JTEXT_DRIVE_ROOT / 'seoul_jtext_split.json'
write_jsonl(TRAIN_JSONL, train_records)
write_jsonl(VAL_JSONL, val_records)
manifest = {
    'version': 1,
    'split_seed': SPLIT_SEED,
    'train_speakers': sorted(train_speakers),
    'val_speakers': sorted(val_speakers),
    'train_rows': len(train_records),
    'val_rows': len(val_records),
    'processing_report': processing_report,
}
SPLIT_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print({'train': len(train_records), 'val': len(val_records)})
print({'train_speakers': len(train_speakers), 'val_speakers': len(val_speakers)})


In [ ]:
# J-text BART와 음절 단위 denoising batch 구성
from torch.utils.data import DataLoader, Dataset
from transformers import BartConfig, BartForConditionalGeneration, BartTokenizer

TOKENIZER_NAME = 'facebook/bart-base'
tokenizer = BartTokenizer.from_pretrained(TOKENIZER_NAME)
base_vocab_size = tokenizer.vocab_size
added_count = tokenizer.add_tokens(list(JAMO_VOCAB))
assert added_count == len(JAMO_VOCAB)
jamo_to_id = {token: tokenizer.convert_tokens_to_ids(token) for token in JAMO_VOCAB}
assert sorted(jamo_to_id.values()) == list(range(base_vocab_size, base_vocab_size + 67))
id_to_jamo = {token_id: token for token, token_id in jamo_to_id.items()}

bart_config = BartConfig.from_pretrained(TOKENIZER_NAME)
jtext_model = BartForConditionalGeneration(bart_config)
jtext_model.resize_token_embeddings(len(tokenizer))
assert jtext_model.config.vocab_size == len(tokenizer)

class SeoulJTextDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        return self.rows[index]

class JamoDenoisingCollator:
    def __init__(self, training, seed):
        self.training = bool(training)
        self.seed = int(seed)

    def _rng(self, row):
        if self.training:
            return random
        key = f"{self.seed}:{row['source_textgrid']}:{row['interval_index']}"
        stable_seed = int(hashlib.sha256(key.encode()).hexdigest()[:16], 16)
        return random.Random(stable_seed)

    def __call__(self, rows):
        sources, targets = [], []
        for row in rows:
            syllables = [list(unicodedata.normalize('NFD', syllable)) for syllable in row['pronounced_hangul']]
            rng = self._rng(row)
            mask_count = max(1, round(len(syllables) * 0.3))
            masked = set(rng.sample(range(len(syllables)), k=min(mask_count, len(syllables))))
            source_ids = [tokenizer.bos_token_id]
            target_ids = [tokenizer.bos_token_id]
            for index, components in enumerate(syllables):
                component_ids = [jamo_to_id[token] for token in components]
                source_ids.extend([tokenizer.mask_token_id] if index in masked else component_ids)
                target_ids.extend(component_ids)
            source_ids.append(tokenizer.eos_token_id)
            target_ids.append(tokenizer.eos_token_id)
            assert len(source_ids) <= MAX_LENGTH and len(target_ids) <= MAX_LENGTH
            sources.append(source_ids)
            targets.append(target_ids)

        source_length = max(map(len, sources))
        target_length = max(map(len, targets))
        input_ids = torch.full((len(rows), source_length), tokenizer.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros_like(input_ids)
        labels = torch.full((len(rows), target_length), -100, dtype=torch.long)
        for row_index, (source_ids, target_ids) in enumerate(zip(sources, targets)):
            input_ids[row_index, :len(source_ids)] = torch.tensor(source_ids)
            attention_mask[row_index, :len(source_ids)] = 1
            labels[row_index, :len(target_ids)] = torch.tensor(target_ids)
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

train_dataset = SeoulJTextDataset(train_records)
val_dataset = SeoulJTextDataset(val_records)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0, collate_fn=JamoDenoisingCollator(True, MODEL_SEED))
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0, collate_fn=JamoDenoisingCollator(False, MODEL_SEED))
batch = next(iter(val_loader))
assert set(batch) == {'input_ids', 'attention_mask', 'labels'}
print('BART base vocabulary:', base_vocab_size)
print('BART + Jamo vocabulary:', len(tokenizer))
print({name: tuple(value.shape) for name, value in batch.items()})


In [ ]:
# 학습, constrained generation, Jamo Error Rate
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import get_cosine_schedule_with_warmup
from transformers.models.bart.modeling_bart import shift_tokens_right

CHOSEONG_IDS = frozenset(jamo_to_id[token] for token in CHOSEONG)
JUNGSEONG_IDS = frozenset(jamo_to_id[token] for token in JUNGSEONG)
JONGSEONG_IDS = frozenset(jamo_to_id[token] for token in JONGSEONG)
SPECIAL_IDS = {tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id, jtext_model.config.decoder_start_token_id}

def allowed_next_tokens(_batch_id, input_ids):
    prefix = [token_id for token_id in input_ids.tolist() if token_id not in {jtext_model.config.decoder_start_token_id, tokenizer.pad_token_id}]
    if not prefix:
        return [tokenizer.bos_token_id]
    if prefix[0] != tokenizer.bos_token_id:
        raise ValueError(f'Generated prefix does not start with BOS: {prefix!r}')
    payload = prefix[1:]
    if not payload:
        return sorted(CHOSEONG_IDS)
    if payload[-1] == tokenizer.eos_token_id:
        return [tokenizer.eos_token_id]
    state = 'choseong'
    completed_syllable = False
    for token_id in payload:
        if state == 'choseong':
            if token_id not in CHOSEONG_IDS:
                raise ValueError(f'Expected choseong, got {token_id}')
            state = 'jungseong'
        elif state == 'jungseong':
            if token_id not in JUNGSEONG_IDS:
                raise ValueError(f'Expected jungseong, got {token_id}')
            state = 'optional_jongseong'
            completed_syllable = True
        elif token_id in JONGSEONG_IDS:
            state = 'choseong'
        elif token_id in CHOSEONG_IDS:
            state = 'jungseong'
        else:
            raise ValueError(f'Invalid Jamo prefix token: {token_id}')
    if state == 'jungseong':
        return sorted(JUNGSEONG_IDS)
    if state == 'optional_jongseong':
        remaining = MAX_LENGTH - len(input_ids)
        if remaining <= 1:
            return [tokenizer.eos_token_id]
        if remaining == 2:
            return sorted(JONGSEONG_IDS | {tokenizer.eos_token_id})
        return sorted(CHOSEONG_IDS | JONGSEONG_IDS | {tokenizer.eos_token_id})
    allowed = set(CHOSEONG_IDS)
    if completed_syllable:
        allowed.add(tokenizer.eos_token_id)
    return sorted(allowed)

def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for ref_token in reference:
        current = [previous[0] + 1]
        for column, hyp_token in enumerate(hypothesis, start=1):
            current.append(min(current[-1] + 1, previous[column] + 1, previous[column - 1] + (ref_token != hyp_token)))
        previous = current
    return previous[-1]

def jamo_ids(sequence):
    return [int(token_id) for token_id in sequence if int(token_id) in id_to_jamo]

def forward_loss(model, batch, device):
    batch = {name: value.to(device, non_blocking=True) for name, value in batch.items()}
    decoder_input_ids = shift_tokens_right(batch['labels'], model.config.pad_token_id, model.config.decoder_start_token_id)
    outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], decoder_input_ids=decoder_input_ids, labels=None, return_dict=True)
    loss = F.cross_entropy(outputs.logits.reshape(-1, outputs.logits.size(-1)), batch['labels'].reshape(-1), ignore_index=-100, label_smoothing=0.1)
    return loss

def train_one_epoch(model, loader, optimizer, scheduler, device, epoch, accumulation_steps):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    progress = tqdm(loader, desc=f'J-text train epoch {epoch}', unit='batch')
    for batch_index, batch in enumerate(progress, start=1):
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = forward_loss(model, batch, device)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite J-text loss: epoch={epoch}, batch={batch_index}')
        (loss / accumulation_steps).backward()
        if batch_index % accumulation_steps == 0 or batch_index == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0, error_if_nonfinite=True)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += float(loss.detach().cpu())
        progress.set_postfix(loss=f'{total_loss / batch_index:.4f}', lr=f"{scheduler.get_last_lr()[0]:.2e}")
    return total_loss / len(loader)

@torch.no_grad()
def validate_loss(model, loader, device):
    model.eval()
    total_loss = 0.0
    for batch in tqdm(loader, desc='J-text validation loss', unit='batch'):
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            total_loss += float(forward_loss(model, batch, device).cpu())
    return total_loss / len(loader)

@torch.no_grad()
def validate_jer(model, loader, device, max_examples=4096):
    model.eval()
    total_edits = 0
    total_reference_tokens = 0
    evaluated = 0
    for batch in tqdm(loader, desc='J-text validation JER', unit='batch'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, num_beams=1, max_length=MAX_LENGTH, prefix_allowed_tokens_fn=allowed_next_tokens)
        for prediction, reference in zip(outputs.cpu().tolist(), batch['labels'].tolist()):
            prediction_tokens = jamo_ids(prediction)
            reference_tokens = jamo_ids(reference)
            total_edits += edit_distance(reference_tokens, prediction_tokens)
            total_reference_tokens += len(reference_tokens)
            evaluated += 1
            if evaluated >= max_examples:
                return total_edits / total_reference_tokens, evaluated
    return total_edits / total_reference_tokens, evaluated


In [ ]:
# J-text 사전학습: tqdm이 batch 진행률과 ETA를 계속 표시한다.
assert torch.cuda.is_available(), 'CUDA GPU가 필요합니다.'
assert torch.cuda.is_bf16_supported(), 'BF16 지원 CUDA GPU가 필요합니다.'
device = 'cuda'
jtext_model = jtext_model.to(device)
JTEXT_EPOCHS = 20
ACCUMULATION_STEPS = 4
EARLY_STOPPING_PATIENCE = 3
JER_MAX_EXAMPLES = 4096
BEST_JTEXT_PATH = JTEXT_DRIVE_ROOT / 'best_jtext_decoder.pt'
LAST_JTEXT_PATH = JTEXT_DRIVE_ROOT / 'last_jtext_training.pt'

optimizer = torch.optim.AdamW(jtext_model.parameters(), lr=1e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-6)
updates_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
total_updates = updates_per_epoch * JTEXT_EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1, round(total_updates * 0.05)), num_training_steps=total_updates)
start_epoch, best_jer, stale_epochs = 1, float('inf'), 0
history = []
if LAST_JTEXT_PATH.is_file():
    resume = torch.load(LAST_JTEXT_PATH, map_location=device, weights_only=False)
    jtext_model.load_state_dict(resume['decoder'])
    optimizer.load_state_dict(resume['optimizer'])
    scheduler.load_state_dict(resume['scheduler'])
    start_epoch = int(resume['epoch']) + 1
    best_jer = float(resume['best_jer'])
    stale_epochs = int(resume['stale_epochs'])
    history = list(resume['history'])
    print('Resume J-text at epoch', start_epoch)

for epoch in range(start_epoch, JTEXT_EPOCHS + 1):
    epoch_started = time.time()
    train_loss = train_one_epoch(jtext_model, train_loader, optimizer, scheduler, device, epoch, ACCUMULATION_STEPS)
    val_loss = validate_loss(jtext_model, val_loader, device)
    val_jer, jer_examples = validate_jer(jtext_model, val_loader, device, JER_MAX_EXAMPLES)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_jer': val_jer, 'jer_examples': jer_examples, 'minutes': (time.time() - epoch_started) / 60}
    history.append(row)
    print(row)
    improved = val_jer < best_jer
    if improved:
        best_jer = val_jer
        stale_epochs = 0
        torch.save({'decoder': {name: value.detach().cpu() for name, value in jtext_model.state_dict().items()}, 'jamo_to_id': jamo_to_id, 'tokenizer_length': len(tokenizer), 'base_vocab_size': base_vocab_size, 'epoch': epoch, 'val_jer': val_jer, 'config': {'model': TOKENIZER_NAME, 'max_length': MAX_LENGTH, 'source_tier_index': 2, 'source_tier_name': 'pWord.prono.', 'phoneme_used': False, 'flac_used': False}, 'split_manifest': str(SPLIT_MANIFEST)}, BEST_JTEXT_PATH)
    else:
        stale_epochs += 1
    torch.save({'decoder': jtext_model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'epoch': epoch, 'best_jer': best_jer, 'stale_epochs': stale_epochs, 'history': history}, LAST_JTEXT_PATH)
    if stale_epochs >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation JER={best_jer:.6f}')
        break

assert BEST_JTEXT_PATH.is_file()
best_jtext = torch.load(BEST_JTEXT_PATH, map_location='cpu', weights_only=False)
assert best_jtext['config']['phoneme_used'] is False
assert best_jtext['config']['flac_used'] is False
display(pd.DataFrame(history))
print('best J-text:', BEST_JTEXT_PATH, 'epoch', best_jtext['epoch'], 'JER', best_jtext['val_jer'])
del jtext_model, optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# OnomaCap canonical-Jamo 데이터 준비
import kagglehub
if str(ONOMAHOW_DIR) not in sys.path:
    sys.path.insert(0, str(ONOMAHOW_DIR))
from jamo_preprocessing import EXPECTED_LATIN_AUDIO, EXPECTED_OUTPUT_ROWS, load_or_create_split_manifest, prepare_rows

audio_root = Path(kagglehub.dataset_download('buraktaci/firat-esc50'))
csv_path = ANNOTATION_DIR / 'sound-to-onomatopoeia_annotation.csv'
with csv_path.open(encoding='utf-8-sig', newline='') as stream:
    prepared_rows, onoma_report = prepare_rows(csv.DictReader(stream))
assert onoma_report.output_rows == EXPECTED_OUTPUT_ROWS == 7_961
assert onoma_report.latin_rows_excluded == (EXPECTED_LATIN_AUDIO,)
df = pd.DataFrame(prepared_rows)

def audio_key(name):
    return unicodedata.normalize('NFKC', Path(str(name)).name).strip().casefold()

extensions = {'.mp3', '.wav', '.flac', '.ogg', '.m4a'}
audio_files = {audio_key(path.name): path for path in audio_root.rglob('*') if path.suffix.lower() in extensions}
df['audio_path'] = df['audio_file'].map(lambda name: str(audio_files.get(audio_key(name), '')))
jamo_columns = [f'candidate{index}_jamo' for index in range(1, 6)]
assert len(df) == 7_961 and df['class'].nunique() == 41
assert not df['audio_path'].eq('').any()
assert df[jamo_columns].map(lambda value: set(value.split()) <= JAMO_SET).all().all()

ONOMA_SPLIT_MANIFEST = Path('/content/drive/MyDrive/OnomaCap/splits/onomacap_7961_seed20_v1.csv')
splits = load_or_create_split_manifest(df, ONOMA_SPLIT_MANIFEST, split_seed=SPLIT_SEED)
assert {name: len(frame) for name, frame in splits.items()} == {'train': 6_368, 'val': 796, 'test': 797}

JSON_DIR = WAVCAPS_DIR / 'captioning/data/OnomaCap/json_files'
JSON_DIR.mkdir(parents=True, exist_ok=True)
for split_name, frame in splits.items():
    data = []
    for _, row in frame.iterrows():
        data.append({'audio': str(Path(row['audio_path']).resolve()), **{f'caption_{index}': str(row[f'candidate{index}_jamo']) for index in range(1, 6)}})
    (JSON_DIR / f'{split_name}.json').write_text(json.dumps({'data': data}, ensure_ascii=False, indent=2), encoding='utf-8')
    print(split_name, len(data))

HTSAT_SOURCE = Path('/content/drive/MyDrive/OnomaCap/pretrained/HTSAT.ckpt')
HTSAT_DESTINATION = WAVCAPS_DIR / 'captioning/pretrained_models/audio_encoder/HTSAT.ckpt'
if not HTSAT_SOURCE.is_file():
    raise FileNotFoundError(f'{HTSAT_SOURCE}가 없습니다.')
HTSAT_DESTINATION.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(HTSAT_SOURCE, HTSAT_DESTINATION)
assert HTSAT_DESTINATION.stat().st_size == HTSAT_SOURCE.stat().st_size


In [ ]:
# J-text transfer 설정과 strict frozen-HTSAT smoke test
import ruamel.yaml as yaml
jamo_yaml = yaml.YAML()
with config_destination.open('r', encoding='utf-8') as stream:
    transfer_config = jamo_yaml.load(stream)
transfer_config['exp_name'] = EXPERIMENT_NAME
transfer_config['seed'] = MODEL_SEED
transfer_config['pretrain'] = False
transfer_config.pop('pretrain_path', None)
transfer_config.pop('pretrain_strict', None)
transfer_config['text_pretrain_path'] = str(BEST_JTEXT_PATH)
transfer_config['audio_encoder_args']['freeze'] = True
transfer_config['audio_encoder_args']['spec_augment'] = False
transfer_config['text_decoder_args']['pretrained'] = False
transfer_config['text_decoder_args']['compact_jamo_vocab'] = False
transfer_config['evaluation']['beam_size'] = EVAL_BEAM_SIZE
with config_destination.open('w', encoding='utf-8') as stream:
    jamo_yaml.dump(transfer_config, stream)

captioning_dir = WAVCAPS_DIR / 'captioning'
previous_cwd = Path.cwd()
smoke_model = None
try:
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))
    from data_handling.datamodule import AudioCaptionDataModule
    from models.bart_captioning import BartCaptionModel
    smoke_data = AudioCaptionDataModule(transfer_config, 'OnomaCap')
    audio, text, _, _ = next(iter(smoke_data.train_dataloader()))
    smoke_model = BartCaptionModel(transfer_config).to('cuda')
    jtext_checkpoint = torch.load(BEST_JTEXT_PATH, map_location='cpu', weights_only=False)
    assert jtext_checkpoint['jamo_to_id'] == smoke_model.jamo_to_id
    assert jtext_checkpoint['tokenizer_length'] == len(smoke_model.tokenizer)
    smoke_model.decoder.load_state_dict(jtext_checkpoint['decoder'], strict=True)
    assert smoke_model.encoder.freeze_audio_encoder is True
    assert not any(parameter.requires_grad for parameter in smoke_model.encoder.parameters())
    smoke_model.train()
    assert smoke_model.encoder.audio_enc.training is False
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        smoke_loss = smoke_model(audio.to('cuda'), text)
    smoke_loss.backward()
    assert all(parameter.grad is None for parameter in smoke_model.encoder.parameters())
    assert any(parameter.grad is not None for parameter in smoke_model.decoder.parameters() if parameter.requires_grad)
    print('J-text transfer smoke loss:', float(smoke_loss.detach().cpu()))
finally:
    os.chdir(previous_cwd)
    if smoke_model is not None:
        del smoke_model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# OnomaCap J-text transfer 학습; train.py의 tqdm이 진행률과 ETA를 표시한다.
assert BEST_JTEXT_PATH.is_file()
os.environ['PYTHONPATH'] = str(captioning_dir)
%cd "{captioning_dir}"
!python train.py --exp_name {EXPERIMENT_NAME} --config settings/onomacap_jamo.yaml --lr 3e-5 --seed {MODEL_SEED}


In [ ]:
# best checkpoint의 full test Jamo Error Rate (추가 지표는 JER만 사용)
from tqdm.auto import tqdm
FOLDER_NAME = f'{EXPERIMENT_NAME}_seed_{MODEL_SEED}'
CHECKPOINT_DIR = Path('/content/drive/MyDrive/OnomaCap/checkpoints') / FOLDER_NAME
BEST_TRANSFER_PATH = CHECKPOINT_DIR / 'best_model.pt'
best_transfer = torch.load(BEST_TRANSFER_PATH, map_location='cpu', weights_only=False)

test_model = BartCaptionModel(transfer_config).to('cuda')
fresh_encoder_state = test_model.encoder.state_dict()
for name, value in fresh_encoder_state.items():
    checkpoint_value = best_transfer['model'][f'encoder.{name}']
    if not torch.equal(value.cpu(), checkpoint_value.cpu()):
        raise RuntimeError(f'Frozen HTSAT changed during transfer: {name}')
test_model.load_state_dict(best_transfer['model'], strict=True)
test_model.eval()
test_loader = AudioCaptionDataModule(transfer_config, 'OnomaCap').test_dataloader()

total_edits = 0
total_reference_tokens = 0
sample_count = 0
with torch.no_grad():
    for batch_data in tqdm(test_loader, desc='OnomaCap test JER', unit='batch'):
        audios, caption_lists, _audio_names, _audio_ids = batch_data
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            predictions = test_model.generate(audios.to('cuda'), num_beams=EVAL_BEAM_SIZE)
        for prediction, references in zip(predictions, caption_lists):
            predicted_tokens = pronounced_hangul_to_jamo(prediction).split()
            reference_token_lists = [reference.split() for reference in references]
            candidates = []
            for reference_tokens in reference_token_lists:
                distance = edit_distance(reference_tokens, predicted_tokens)
                candidates.append((distance / len(reference_tokens), distance, len(reference_tokens)))
            _, selected_distance, selected_length = min(candidates)
            total_edits += selected_distance
            total_reference_tokens += selected_length
            sample_count += 1

test_jer = total_edits / total_reference_tokens
jer_result = {
    'checkpoint': str(BEST_TRANSFER_PATH),
    'epoch': int(best_transfer['epoch']),
    'beam_size': EVAL_BEAM_SIZE,
    'samples': sample_count,
    'total_edits': total_edits,
    'total_reference_tokens': total_reference_tokens,
    'jamo_error_rate': test_jer,
    'reference_policy': 'minimum normalized JER among five references',
}
JER_RESULT_PATH = Path('/content/drive/MyDrive/OnomaCap/results') / FOLDER_NAME / 'test_jer.json'
JER_RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
JER_RESULT_PATH.write_text(json.dumps(jer_result, ensure_ascii=False, indent=2), encoding='utf-8')
display(jer_result)
print('JER result:', JER_RESULT_PATH)
